In [1]:
import xarray as xr
import glob
import numpy as np
import matplotlib.pyplot as plt
import pymannkendall as mk
import pandas as pd

In [ ]:
files = sorted(glob.glob("../disdata/finn_data/*.nc"))

# Bounding boxes (0-360 lon convention)
core = dict(lat=slice(-39, -36), lon=slice(286, 289))

core_list = []
for fp in files:
    ds = xr.open_dataset(fp)
    
    if "fire_modis_PM25" in ds:
        ds = ds.rename({"fire_modis_PM25": "PM25"})
        ds["sensor"] = "modis"
    elif "fire_modisviirs_PM25" in ds:
        ds = ds.rename({"fire_modisviirs_PM25": "PM25"})
        ds["sensor"] = "modisviirs"
    
    ds = ds.drop_vars("date")
    
    core_list.append(ds.sel(**core))

ds_core = xr.concat(core_list, dim="time")

# Restrict to the formal comparison period: fire year 2003/04 to 2022/23
ds_core = ds_core.sel(time=slice("2003-07-01", "2023-06-30"))

print(ds_core)
print(ds_core.data_vars)
print(ds_core.variables.keys())

In [ ]:
for name, ds in [("core", ds_core)]:
    pm25 = ds["PM25"]
    print(f"\n--- {name} ---")
    print(f"  shape      : {pm25.shape}")
    print(f"  total cells: {pm25.size}")
    print(f"  non-NaN    : {int(pm25.notnull().sum())}")
    print(f"  all-zero   : {bool((pm25.fillna(0) == 0).all())}")
    print(f"  max        : {float(pm25.max()):.4g}")
    print(f"  mean (non-NaN): {float(pm25.mean()):.4g}")

In [ ]:
print(ds_core["PM25"].attrs)

In [ ]:
# Constants
N_AV = 6.022e23      # molecules/mol
MW_PM25 = 12         #  g/mol, per NCAR documentation for FINNv2.5 gridded files
                     # (aerosols converted from kg to molecules using MW=12 g/mol)
SECS_PER_DAY = 86400
GRID_AREA_M2 = (0.1 * 111000) ** 2  # approximate area of 0.1° cell in m²

def convert_to_kg_per_day(ds):
    """Convert PM25 from molecules/cm2/s to kg/day per grid cell."""
    pm25_kg = (
        ds["PM25"]
        * 1e4              # cm2 to m2
        / N_AV             # molecules to moles
        * MW_PM25          # g/mol to g
        / 1000             # g to kg
        * SECS_PER_DAY     # per second to per day
        * GRID_AREA_M2     # per m2 to per grid cell
    )
    ds["PM25_kg_day"] = pm25_kg
    ds["PM25_kg_day"].attrs["units"] = "kg/day/cell"
    return ds

ds_core     = convert_to_kg_per_day(ds_core)

# Sanity check
print(ds_core["PM25_kg_day"].max().values)
print(ds_core["PM25_kg_day"].mean().values)

In [ ]:
LAT_CUTOFF = -38.0

biobio_mask = ds_core["lat"] > LAT_CUTOFF
araucania_mask = ds_core["lat"] <= LAT_CUTOFF

biobio_finn = ds_core["PM25_kg_day"].where(biobio_mask, drop=True).sum(dim=["lat", "lon"]).rename("PM25_kg_day_sum")
araucania_finn = ds_core["PM25_kg_day"].where(araucania_mask, drop=True).sum(dim=["lat", "lon"]).rename("PM25_kg_day_sum")

biobio_finn_df = biobio_finn.to_dataframe().reset_index()
araucania_finn_df = araucania_finn.to_dataframe().reset_index()

biobio_finn_df["region"] = "VIII"
araucania_finn_df["region"] = "IX"

finn_regional = pd.concat([biobio_finn_df, araucania_finn_df], ignore_index=True)
finn_regional = finn_regional.rename(columns={"time": "date_only"})
finn_regional["date_only"] = pd.to_datetime(finn_regional["date_only"]).dt.normalize()

finn_regional.to_csv("../disdata/sinca_finn_comparison/finn_daily_regional.csv", index=False)

print(finn_regional.groupby("region")["PM25_kg_day_sum"].describe())

In [ ]:
# Summarise PM25_kg_day across all grid cells for each day
ts_core = ds_core["PM25_kg_day"].sum(dim=["lat", "lon"]).rename("PM25_kg_day_sum")
ts_core_mean = ds_core["PM25_kg_day"].mean(dim=["lat", "lon"]).rename("PM25_kg_day_mean")

# Combine into dataframes
df_core = pd.DataFrame({
    "PM25_kg_day_sum": ts_core.values,
    "PM25_kg_day_mean": ts_core_mean.values,
}, index=ts_core.time.values)

df_core.index.name = "date"
print(df_core.head(10))

In [ ]:
plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.labelsize": 13,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
})

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
for ax, metric, col in zip(
    axes.flat,
    ["Sum", "Mean"],
    ["PM25_kg_day_sum", "PM25_kg_day_mean"]
):
    ax.plot(df_core.index, df_core[col], linewidth=0.6, color="darkred")
    ax.set_title(f"Daily {metric}")
    ax.set_ylabel("kg/day")
    ax.grid(True, linewidth=0.3)
axes[1].set_xlabel("Date")
fig.suptitle("FINN v2.5 PM₂.₅ Daily Emissions Time Series", fontsize=18)
fig.autofmt_xdate()
plt.tight_layout()
plt.savefig("../outputs/finn_pm25_timeseries.png", dpi=150)
plt.show()

In [ ]:
print(df_core["PM25_kg_day_sum"].describe())

In [ ]:
# Assign fire year to daily data
df_core.index = pd.to_datetime(df_core.index)
df_core["fire_year"] = df_core.index.map(
    lambda d: d.year if d.month >= 7 else d.year - 1
)

# Annual means
finn_annual = df_core.groupby("fire_year").agg(
    PM25_kg_day_sum_mean=("PM25_kg_day_sum", "mean"),
    PM25_kg_day_mean_mean=("PM25_kg_day_mean", "mean")
).reset_index()

print(finn_annual)

In [ ]:
# restrict to fire years from 2002 onward (i.e. the 2002/03 fire season onward)
finn_annual_2002 = finn_annual[finn_annual["fire_year"] >= 2002].copy()

# Mann-Kendall
for metric, col in [
    ("Daily sum (mean per fire year)", "PM25_kg_day_sum_mean"),
    ("Daily mean (mean per fire year)", "PM25_kg_day_mean_mean")
]:
    result = mk.original_test(finn_annual_2002[col].values)
    print(f"\n--- {metric} ---")
    print(f"  Trend      : {result.trend}")
    print(f"  p-value    : {result.p:.4f}")
    print(f"  Sen's slope: {result.slope:.4e} kg/day per year")
    print(f"  Tau        : {result.Tau:.4f}")

# Sen's slope trend line
from scipy.stats import theilslopes

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
for ax, col, metric in zip(
    axes.flat,
    ["PM25_kg_day_sum_mean", "PM25_kg_day_mean_mean"],
    ["Daily Sum (mean per fire year)", "Daily Mean (mean per fire year)"]
):
    x = finn_annual_2002["fire_year"].values
    
    if col == "PM25_kg_day_sum_mean":
        y = finn_annual_2002[col].values / 1e4
        ylabel = "PM$_{2.5}$ (×10$^{4}$ kg/day)"
        slope_unit = "×10⁴ kg/day/yr"
    else:
        y = finn_annual_2002[col].values
        ylabel = "PM$_{2.5}$ (kg/day)"
        slope_unit = "kg/day/yr"
    
    # Sen's slope line (fit stays on numeric x)
    slope, intercept, _, _ = theilslopes(y, x)
    trend_line = intercept + slope * x
    ax.plot(x, y, color="darkred", linewidth=1.5, marker="o", markersize=4, label="Annual mean")
    ax.plot(x, trend_line, color="darkgrey", linewidth=1.2, linestyle="--", label=f"Sen's slope: {slope:.2e} {slope_unit}")
    ax.set_ylabel(ylabel)
    ax.set_title(f"PM₂.₅ {metric}")
    ax.legend(fontsize=9)
    ax.grid(True, linewidth=0.3)

# build fire-year-range labels (e.g. 2002-2003) and apply every other year
fire_year_labels = [f"{yr}-{yr+1}" for yr in x]
tick_positions = x[::2]
tick_labels = fire_year_labels[::2]

axes[1].set_xticks(tick_positions)
axes[1].set_xticklabels(tick_labels)
axes[1].set_xlabel("Fire year")
plt.setp(axes[1].get_xticklabels(), rotation=45, ha="right")

fig.suptitle("FINN v2.5 PM₂.₅ Annual Trend")
plt.tight_layout()
plt.savefig("../outputs/fig8_finn_pm25_timeseries_trend.png", dpi=400, bbox_inches="tight")
plt.show()

In [ ]:
# restrict to fire years from 2002 onward (i.e. the 2002/03 fire season onward)
finn_annual_2002 = finn_annual[finn_annual["fire_year"] >= 2002].copy()

# Mann-Kendall
result = mk.original_test(finn_annual_2002["PM25_kg_day_sum_mean"].values)
print(f"\n--- Daily sum (mean per fire year) ---")
print(f"  Trend      : {result.trend}")
print(f"  p-value    : {result.p:.4f}")
print(f"  Sen's slope: {result.slope:.4e} kg/day per year")
print(f"  Tau        : {result.Tau:.4f}")

# Sen's slope trend line
from scipy.stats import theilslopes

fig, ax = plt.subplots(figsize=(12, 5))

x = finn_annual_2002["fire_year"].values
y = finn_annual_2002["PM25_kg_day_sum_mean"].values / 1e4
ylabel = "PM$_{2.5}$ (×10$^{4}$ kg/day)"
slope_unit = "×10⁴ kg/day/yr"

# Sen's slope line (fit stays on numeric x)
slope, intercept, _, _ = theilslopes(y, x)
trend_line = intercept + slope * x
ax.plot(x, y, color="darkred", linewidth=1.5, marker="o", markersize=4, label="Annual mean")
ax.plot(x, trend_line, color="dimgrey", linewidth=1.2, linestyle="--", label=f"Sen's slope: {slope:.2e} {slope_unit}")
ax.set_ylabel(ylabel)
ax.set_title("PM₂.₅ Daily Sum (mean per fire year)")
ax.legend(fontsize=9)
ax.grid(True, linewidth=0.3)

# build fire-year-range labels (e.g. 2002-2003) and show every year on the x-axis
fire_year_labels = [f"{yr}-{yr+1}" for yr in x]
ax.set_xticks(x)
ax.set_xticklabels(fire_year_labels, rotation=45, ha="right")
ax.set_xlabel("Fire year")

plt.tight_layout()
plt.savefig("../outputs/finn_pm25_timeseries_dailysum.png", dpi=400, bbox_inches="tight")
plt.show()

## Corrected Saved output for FINN data after fixing the units

In [ ]:
df_core_reset = df_core.reset_index()
df_core_reset.to_csv("../outputs/finn_pm25_daily_core.csv", index=False)
print(f"Saved {len(df_core_reset)} rows to finn_pm25_daily_core.csv")

## Investigating Spikes

In [ ]:
print(df_core.columns.tolist())
print(df_core.head())
df_core["month"] = df_core.index.month

In [ ]:
spike_years = [2014, 2016, 2022]  # fire_year start values for 2014/15, 2016/17, 2022/23

# z-scores, using the same metric as your trend plot (daily sum, mean per fire year)
mean_val = finn_annual["PM25_kg_day_sum_mean"].mean()
std_val = finn_annual["PM25_kg_day_sum_mean"].std()
finn_annual["z_score"] = (finn_annual["PM25_kg_day_sum_mean"] - mean_val) / std_val

spike_summary = finn_annual[finn_annual["fire_year"].isin(spike_years)].copy()
spike_summary["fire_year_label"] = spike_summary["fire_year"].apply(lambda y: f"{y}-{y+1}")
spike_summary["pct_above_mean"] = (spike_summary["PM25_kg_day_sum_mean"] - mean_val) / mean_val * 100
print(spike_summary[["fire_year_label", "PM25_kg_day_sum_mean", "z_score", "pct_above_mean"]])

In [ ]:
# monthly breakdown within each spike year 
for year in spike_years:
    year_data = df_core[df_core["fire_year"] == year]
    monthly = year_data.groupby("month")["PM25_kg_day_sum"].sum().reindex(
        [7, 8, 9, 10, 11, 12, 1, 2, 3, 4, 5, 6], fill_value=0
    )
    print(f"\n--- Fire year {year}-{year+1} monthly totals (kg) ---")
    print(monthly)

In [ ]:
# peak single-day emissions within each spike year 
for year in spike_years:
    year_data = df_core[df_core["fire_year"] == year]
    top_days = year_data.nlargest(5, "PM25_kg_day_sum")["PM25_kg_day_sum"]
    print(f"\n--- Fire year {year}-{year+1} top 5 single-day emissions ---")
    print(top_days)